# BirdCLEF+ 2026 EDA

Macro-averaged ROC-AUC over 234 species, CPU-only inference (90min limit).
Audio from Pantanal, Brazil — 5-second chunk prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)

BASE = '../data'

train = pd.read_csv(f'{BASE}/train.csv')
taxonomy = pd.read_csv(f'{BASE}/taxonomy.csv')
sample_sub = pd.read_csv(f'{BASE}/sample_submission.csv')

print('Train shape:', train.shape)
print('Taxonomy shape:', taxonomy.shape)
print('Sample submission shape:', sample_sub.shape)

In [ ]:
# Basic stats
print('Train columns:', train.columns.tolist())
train.head(3)

In [ ]:
# Species (class) distribution
species_col = 'primary_label' if 'primary_label' in train.columns else 'species_id'
print('Unique species:', train[species_col].nunique())

counts = train[species_col].value_counts()
print(f'Min samples per species: {counts.min()}')
print(f'Max samples per species: {counts.max()}')
print(f'Median samples per species: {counts.median():.0f}')

plt.figure(figsize=(14,4))
plt.bar(range(len(counts)), sorted(counts.values, reverse=True))
plt.xlabel('Species (sorted by count)')
plt.ylabel('Number of recordings')
plt.title('Per-species recording counts (long-tail distribution)')
plt.tight_layout()
plt.show()

In [ ]:
# Taxonomy overview
print('Taxonomy columns:', taxonomy.columns.tolist())
taxonomy.head(5)

In [ ]:
# Sample submission structure — shows prediction format
print('Submission columns (first 5):', sample_sub.columns[:5].tolist())
print('Total columns:', len(sample_sub.columns))
print('Total rows:', len(sample_sub))
sample_sub.head(3)

In [ ]:
# Duration distribution (if available)
if 'duration' in train.columns:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    train['duration'].hist(bins=50)
    plt.xlabel('Duration (s)')
    plt.title('Recording Duration Distribution')
    
    plt.subplot(1,2,2)
    plt.boxplot(train['duration'])
    plt.title('Duration Boxplot')
    plt.tight_layout()
    plt.show()
    print(train['duration'].describe())

In [ ]:
# Low-data species (important for focal loss / oversampling strategy)
rare = counts[counts < 5]
print(f'Species with < 5 recordings: {len(rare)} ({len(rare)/len(counts)*100:.1f}%)')

# Top 10 most common species
print('\nTop 10 most common:')
print(counts.head(10))

In [ ]:
# Audio file path analysis
if 'filename' in train.columns:
    path_col = 'filename'
elif 'filepath' in train.columns:
    path_col = 'filepath'
else:
    path_col = None
    
if path_col:
    import os
    train_audio_dir = f'{BASE}/train_audio'
    sample_files = train[path_col].head(5).tolist()
    print('Sample file paths:')
    for f in sample_files:
        full = os.path.join(train_audio_dir, f) if not f.startswith('/') else f
        exists = os.path.exists(full)
        print(f'  {f} -> exists={exists}')